In [ ]:
import os
import random
import json
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter

class AdvancedReceiptGenerator:
    def __init__(self, output_dir="synth_data", fonts_dir="fonts"):
        self.output_dir = output_dir
        self.fonts_dir = fonts_dir
        # Создаем структуру папок как в датасете
        self.images_dir = os.path.join(output_dir, "images")
        os.makedirs(self.images_dir, exist_ok=True)
        
        self.font_files = []
        if os.path.exists(fonts_dir):
            self.font_files = [os.path.join(fonts_dir, f) for f in os.listdir(fonts_dir) 
                              if f.endswith(('.ttf', '.otf'))]
        
        if not self.font_files:
            print("⚠️ Шрифты не найдены. Используется стандартный шрифт.")

    def get_random_font(self, size=24):
        if self.font_files:
            return ImageFont.truetype(random.choice(self.font_files), size)
        return ImageFont.load_default()

    def generate_random_data(self):
        """Создает случайный набор данных для чека"""
        companies = ["99 SPEED MART", "SEN LEE HEONG", "HENG KEE DELIGHTS", "GRAND COMPANIONS", "PINTAR SDN BHD"]
        addresses = [
            "LOT P.T. 2811, JALAN BALAKONG", 
            "G-0-1, M.AVENUE NO.1, JLN 1/38A",
            "LOT G112, AEON MALL TEBRAU CITY",
            "NO31, JALAN KLANG LAMA",
            "52, JALAN CHOW KIT, K.LUMPUR"
        ]
        day, month, year = random.randint(1, 28), random.randint(1, 12), random.randint(2017, 2022)
        total = round(random.uniform(1.0, 300.0), 2)
        cash = round(total + random.uniform(5, 50), 0)
        change = round(cash - total, 2)

        return {
            "company": random.choice(companies),
            "address": random.choice(addresses),
            "date": f"{day:02d}-{month:02d}-{year}",
            "total": f"{total:.2f}",
            "cash": f"{cash:.2f}",
            "change": f"{change:.2f}"
        }

    # --- СЕКЦИЯ АУГМЕНТАЦИЙ ---

    def add_noise(self, pil_img):
        """Добавление зернистости (шум сенсора)"""
        img = np.array(pil_img)
        noise = np.random.normal(0, random.randint(5, 15), img.shape).astype(np.uint8)
        img = cv2.add(img, noise)
        return Image.fromarray(img)

    def add_motion_blur(self, pil_img):
        """Имитация смазанного кадра (движение рук)"""
        img = np.array(pil_img)
        size = random.choice([3, 5])
        kernel = np.zeros((size, size))
        # Горизонтальное или вертикальное размытие
        if random.random() > 0.5:
            kernel[int((size-1)/2), :] = np.ones(size)
        else:
            kernel[:, int((size-1)/2)] = np.ones(size)
        kernel = kernel / size
        img = cv2.filter2D(img, -1, kernel)
        return Image.fromarray(img)

    def degrade_ink(self, pil_img):
        """Имитация плохой печати/выцветания красок"""
        img = np.array(pil_img)
        # Создаем маску случайных "пропусков" в печати
        mask = np.random.choice([0, 255], size=img.shape[:2], p=[0.02, 0.98]).astype(np.uint8)
        mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)
        # Там где 0 в маске — заменяем на цвет фона (светло-серый/белый)
        img[mask == 0] = 240 
        return Image.fromarray(img)

    def apply_distortions(self, pil_img):
        """Применяет цепочку искажений"""
        # 1. Размытие (фокус или движение)
        if random.random() > 0.5:
            if random.random() > 0.5:
                pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 0.8)))
            else:
                pil_img = self.add_motion_blur(pil_img)
        
        # 2. Деградация чернил
        if random.random() > 0.7:
            pil_img = self.degrade_ink(pil_img)

        # 3. Цифровой шум
        pil_img = self.add_noise(pil_img)
        
        return pil_img

    def create_image(self, data, idx):
        """Рисует синтетический чек с искажениями"""
        # Фон с легким оттенком (не идеально белый)
        bg_val = random.randint(245, 255)
        img = Image.new('RGB', (600, 1000), color=(bg_val, bg_val, bg_val))
        d = ImageDraw.Draw(img)
        
        title_font = self.get_random_font(32)
        main_font = self.get_random_font(24)

        curr_y = 50
        # Компания
        d.text((300, curr_y), data['company'], fill=(30,30,30), font=title_font, anchor="mm")
        
        curr_y += 60
        # Адрес
        d.text((300, curr_y), data['address'], fill=(60,60,60), font=main_font, anchor="mm")
        
        curr_y += 100
        d.text((50, curr_y), f"DATE: {data['date']}", fill=(40,40,40), font=main_font)
        
        curr_y += 200
        # Блок TOTAL (самый важный для инференса)
        d.text((50, curr_y), f"TOTAL:   RM {data['total']}", fill=(10,10,10), font=title_font)
        curr_y += 50
        d.text((50, curr_y), f"CASH:    RM {data['cash']}", fill=(40,40,40), font=main_font)
        curr_y += 40
        d.text((50, curr_y), f"CHANGE:  RM {data['change']}", fill=(40,40,40), font=main_font)

        # ПРИМЕНЯЕМ ФИЗИЧЕСКИЕ ИСКАЖЕНИЯ
        img = self.apply_distortions(img)

        # Сохранение
        img_name = f"synth_{idx}.jpg"
        img.save(os.path.join(self.images_dir, img_name), quality=random.randint(60, 90))
        return img_name

# Запуск генерации
gen = AdvancedReceiptGenerator(fonts_dir="./my_fonts")
manifest = []

num_samples = 1000 # Рекомендую 1000 для заметного эффекта
print(f"🚀 Начинаю генерацию {num_samples} образцов...")

for i in range(num_samples):
    data = gen.generate_random_data()
    img_name = gen.create_image(data, i)
    # Формат metadata.jsonl для Donut (SROIE style)
    manifest.append({
        "file_name": f"images/{img_name}",
        "ground_truth": json.dumps({"gt_parse": data}, ensure_ascii=False)
    })

# Запись метаданных
with open(os.path.join("synth_data", "metadata.jsonl"), 'w', encoding='utf-8') as f:
    for entry in manifest:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"✨ Готово! Данные сохранены в папку {gen.output_dir}")

✨ Готово! Не забудь добавить аугментации при загрузке этих картинок.


In [ ]:
import json
import os

def convert_jsonl_to_sroie_txt(jsonl_path, output_dir):
    """
    Разбивает metadata.jsonl на отдельные .txt файлы для каждого изображения.
    Формат внутри .txt: чистый JSON объект с полями (company, date, address, total).
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"[*] Создана папка для аннотаций: {output_dir}")

    count = 0
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            try:
                # Читаем строку из metadata.jsonl
                data = json.loads(line)
                
                # file_name может быть 'images/synth_0.jpg', нам нужно только 'synth_0'
                base_name = os.path.basename(data['file_name'])
                file_id = os.path.splitext(base_name)[0]
                
                # Вытаскиваем структуру gt_parse
                # ground_truth в jsonl обычно записан как строка, поэтому парсим дважды
                gt_content = json.loads(data['ground_truth'])
                actual_data = gt_content['gt_parse']
                
                # Формируем путь к .txt файлу (например, synth_0.txt)
                output_path = os.path.join(output_dir, f"{file_id}.txt")
                
                # Сохраняем данные как JSON-строку внутри текстового файла
                with open(output_path, 'w', encoding='utf-8') as out_f:
                    # SROIE формат обычно требует компактный или indent=4 JSON
                    json.dump(actual_data, out_f, ensure_ascii=False, indent=4)
                
                count += 1
            except Exception as e:
                print(f"[!] Ошибка при обработке строки: {e}")

    print(f"✅ Готово! Создано {count} .txt файлов в папке: {output_dir}")

# Настройки путей
JSONL_INPUT = "synth_data/metadata.jsonl"
OUTPUT_FOLDER = "synth_data/txt_annotations" # Можно заменить на путь к папке 'entities' или 'box'

if os.path.exists(JSONL_INPUT):
    convert_jsonl_to_sroie_txt(JSONL_INPUT, OUTPUT_FOLDER)
else:
    print(f"[-] Файл {JSONL_INPUT} не найден. Сначала запусти генератор синтетики.")

Создана папка: synth_data/entities
✅ Готово! Сконвертировано 500 файлов в папку synth_data/entities
